# Lab 3 - hinet_lite_npu_v1 (Colab v3)

- Author: <your_name>
- Date: 2026-04-23
- Goal: improve PSNR with a norm-free mixed-kernel residual U-Net while preserving the Lab 3 artifact chain
- Expected input/output: 256x256x3

This notebook is self-contained for Lab 3: setup, paired data loading, model training, validation, ONNX export, calibration export, and MXQ handoff metadata.


In [2]:
try:
    import google.colab  # type: ignore
    IN_COLAB_BOOTSTRAP = True
except Exception:
    IN_COLAB_BOOTSTRAP = False

if IN_COLAB_BOOTSTRAP:
    %pip install -q onnx

import csv
import json
import math
import os
import random
import shutil
import sys
import time
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import onnx
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


try:
    BICUBIC_RESAMPLE = Image.Resampling.BICUBIC
except AttributeError:
    BICUBIC_RESAMPLE = Image.BICUBIC

try:
    FLIP_LEFT_RIGHT = Image.Transpose.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.Transpose.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.Transpose.ROTATE_90
    ROTATE_180 = Image.Transpose.ROTATE_180
    ROTATE_270 = Image.Transpose.ROTATE_270
except AttributeError:
    FLIP_LEFT_RIGHT = Image.FLIP_LEFT_RIGHT
    FLIP_TOP_BOTTOM = Image.FLIP_TOP_BOTTOM
    ROTATE_90 = Image.ROTATE_90
    ROTATE_180 = Image.ROTATE_180
    ROTATE_270 = Image.ROTATE_270


def now_stamp() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def is_colab_runtime() -> bool:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def auto_num_workers(in_colab: bool) -> int:
    if sys.platform == "darwin":
        return 0
    cpu_count = os.cpu_count() or 2
    if in_colab:
        return min(4, max(cpu_count - 1, 2))
    return min(2, cpu_count)


def resolve_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Data").exists() and (candidate / "runs").exists():
            return candidate
    for candidate in [current, *current.parents]:
        if (candidate / "Data").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root from {start.resolve()}")


def find_latest_model_checkpoint(runs_root: Path, model_id: str) -> Path:
    candidates: list[Path] = []
    for pattern in (f"**/{model_id}*/checkpoints/last.pth", f"**/{model_id}*/checkpoints/latest.pth"):
        candidates.extend(path for path in runs_root.glob(pattern) if path.is_file())
    if not candidates:
        raise FileNotFoundError(f"No prior checkpoint found under {runs_root} for model_id={model_id}")
    return max(candidates, key=lambda path: path.stat().st_mtime)


def resolve_resume_checkpoint_path(resume_checkpoint_path: str | None, runs_root: Path, model_id: str) -> str | None:
    if resume_checkpoint_path is None:
        return None
    if str(resume_checkpoint_path).lower() == "auto":
        return str(find_latest_model_checkpoint(runs_root, model_id))
    path = Path(resume_checkpoint_path).expanduser()
    if not path.is_absolute():
        path = runs_root / path
    if not path.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {path}")
    return str(path)




def checkpoint_psnr_metadata(checkpoint_path: str | None) -> dict[str, Any] | None:
    if not checkpoint_path:
        return None
    path = Path(checkpoint_path)
    if not path.exists():
        return {"path": str(path), "exists": False}
    checkpoint = torch.load(path, map_location="cpu")
    last_metrics = checkpoint.get("last_metrics") or {}
    return {
        "path": str(path),
        "exists": True,
        "epoch": checkpoint.get("epoch"),
        "last_val_psnr": last_metrics.get("val_psnr"),
        "last_delta_psnr": last_metrics.get("delta_psnr"),
        "best_val_psnr": checkpoint.get("best_val_psnr"),
        "best_epoch": checkpoint.get("best_epoch"),
        "selected_weight_source": checkpoint.get("selected_weight_source"),
    }


def print_resume_checkpoint_psnr_summary(resume_checkpoint_path: str | None) -> None:
    last_info = checkpoint_psnr_metadata(resume_checkpoint_path)
    best_path = None
    if resume_checkpoint_path:
        checkpoint_path = Path(resume_checkpoint_path)
        if checkpoint_path.parent.name == "checkpoints":
            best_path = checkpoint_path.parent / "best.pth"
        else:
            best_path = checkpoint_path
    best_info = checkpoint_psnr_metadata(str(best_path)) if best_path else None
    print({
        "resume_last_checkpoint_psnr": last_info,
        "resume_best_checkpoint_psnr": best_info,
    })

def make_grad_scaler(enabled: bool):
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_context(device_type: str, enabled: bool):
    if not enabled:
        return nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast(device_type=device_type, enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)


IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp"}
EXPECTED_TRAIN_PAIRS = 2217
EXPECTED_VAL_PAIRS = 110
DRIVE_ROOT = Path("/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3")
DRIVE_DATA_ROOT = DRIVE_ROOT / "Data"
DRIVE_RUNS_ROOT = DRIVE_ROOT / "runs"


@dataclass
class RunConfig:
    model_id: str = "hinet_lite_npu_v1"
    experiment_tag: str = "colab_v3"
    run_mode: str = "full"
    seed: int = 1337
    train_patch_size: int = 224
    eval_size: int = 256
    batch_size: int = 8
    epochs: int = 160
    lr: float = 3e-4
    min_lr: float = 1e-5
    weight_decay: float = 5e-5
    warmup_epochs: int = 5
    grad_clip_norm: float = 1.0
    num_workers: int | None = None
    prefetch_factor: int = 2
    train_pair_limit: int | None = None
    val_pair_limit: int | None = None
    use_amp: bool = True
    use_ema: bool = True
    use_channels_last: bool = True
    use_tf32: bool = True
    ema_decay: float = 0.999
    train_eval_subset_size: int = 128
    enable_resume: bool = True
    resume_checkpoint_path: str | None = "20260423/hinet_lite_npu_v1_colab_v3_full_20260423_234240/checkpoints/last.pth"
    resume_strict: bool = True
    selected_weight_source: str = "ema"
    stop_if_no_resume_best_after_epochs: int | None = 10


cfg = RunConfig()
assert cfg.eval_size == 256, "Lab 3 eval size must stay 256"
assert cfg.run_mode == "full", "This notebook is full-mode only"
assert cfg.selected_weight_source in {"ema", "raw"}
if cfg.selected_weight_source == "ema" and not cfg.use_ema:
    raise ValueError("selected_weight_source='ema' requires use_ema=True")

in_colab = is_colab_runtime()
if in_colab:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive", force_remount=False)
    data_root = DRIVE_DATA_ROOT
    runs_root = DRIVE_RUNS_ROOT
    if not data_root.exists():
        raise FileNotFoundError(f"Google Drive data root not found: {data_root}")
else:
    repo_root = resolve_repo_root(Path.cwd())
    data_root = repo_root / "Data"
    runs_root = repo_root / "runs"
    if not data_root.exists():
        raise FileNotFoundError(f"Data root not found: {data_root}")

if cfg.num_workers is None:
    cfg.num_workers = auto_num_workers(in_colab)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    if cfg.use_tf32:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        if hasattr(torch, "set_float32_matmul_precision"):
            torch.set_float32_matmul_precision("high")

started_day = datetime.now().strftime("%Y%m%d")
run_name = f"{cfg.model_id}_{cfg.experiment_tag}_{cfg.run_mode}_{now_stamp()}"
run_root = runs_root / started_day / run_name
checkpoints_dir = run_root / "checkpoints"
exports_dir = run_root / "exports"
calibration_dir = exports_dir / "calibration"
summary_path = run_root / "summary.json"
metrics_csv_path = run_root / "metrics.csv"
epoch_logs_jsonl_path = run_root / "epoch_logs.jsonl"
history_json_path = run_root / "history.json"

for directory in [run_root, checkpoints_dir, exports_dir, calibration_dir]:
    directory.mkdir(parents=True, exist_ok=True)

set_global_seed(cfg.seed)
if cfg.enable_resume:
    cfg.resume_checkpoint_path = resolve_resume_checkpoint_path(cfg.resume_checkpoint_path, runs_root, cfg.model_id)
startup_mode = "resume_run" if cfg.enable_resume else "fresh_run"
if cfg.enable_resume and not cfg.resume_checkpoint_path:
    raise ValueError("enable_resume=True requires resume_checkpoint_path or 'auto'")
print(f"startup_mode={startup_mode}")
print({
    "device": str(device),
    "experiment_tag": cfg.experiment_tag,
    "run_mode": cfg.run_mode,
    "in_colab": in_colab,
    "data_root": str(data_root),
    "run_root": str(run_root),
    "num_workers": cfg.num_workers,
    "prefetch_factor": cfg.prefetch_factor,
    "enable_resume": cfg.enable_resume,
    "resume_checkpoint_path": cfg.resume_checkpoint_path,
    "selected_weight_source": cfg.selected_weight_source,
})
print_resume_checkpoint_psnr_summary(cfg.resume_checkpoint_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
startup_mode=resume_run
{'device': 'cuda', 'experiment_tag': 'colab_v3', 'run_mode': 'full', 'in_colab': True, 'data_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/Data', 'run_root': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260424/hinet_lite_npu_v1_colab_v3_full_20260424_004323', 'num_workers': 4, 'prefetch_factor': 2, 'enable_resume': True, 'resume_checkpoint_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_colab_v3_full_20260423_234240/checkpoints/last.pth', 'selected_weight_source': 'ema'}
{'resume_last_checkpoint_psnr': {'path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_colab_v3_full_20260423_234240/checkpoints/last.pth', 'exists': True, 'epoch': 33, 'last_val_psnr': 23.83221231188093, 'l

## 1. Setup

Colab/runtime setup, fixed Lab 3 configuration, and stable artifact paths.

## 2. Data

Canonical LR/HR pairing, synchronized `224x224` train augmentation, fixed full-image train-eval subset, and baseline validation PSNR readout.


In [3]:
EXTRACT_DATASET_ZIP = False
DATASET_ZIP_PATH = data_root / "dataset.zip"
L3_TRAIN_SPLITS = (
    ("LR_HR/L3_HR_train1", "L3_LR/L3_L3_train1", "L3_HR_train1"),
    ("LR_HR/L3_HR_train2", "L3_LR/L3_L3_train2", "L3_HR_train2"),
    ("LR_HR/L3_HR_train3", "L3_LR/L3_LR_train3", "L3_HR_train3"),
    ("LR_HR/L3_HR_train4", "L3_LR/L3_LR_train4", "L3_HR_train4"),
    ("LR_HR/L3_HR_train", "L3_LR/L3_LR_train", "L3_HR_train"),
)
L3_VAL_DIRS = ("L3_HR_valid", "L3_LR_valid")

required_dirs = [data_root / hr_rel for hr_rel, _, _ in L3_TRAIN_SPLITS] + [
    data_root / lr_rel for _, lr_rel, _ in L3_TRAIN_SPLITS
] + [
    data_root / L3_VAL_DIRS[0],
    data_root / L3_VAL_DIRS[1],
]
if EXTRACT_DATASET_ZIP:
    import zipfile

    if not DATASET_ZIP_PATH.exists():
        raise FileNotFoundError(f"Dataset zip not found: {DATASET_ZIP_PATH}")
    with zipfile.ZipFile(DATASET_ZIP_PATH, "r") as zf:
        zf.extractall(data_root)
    print(f"Extracted dataset archive into {data_root}")
missing_dirs = [p for p in required_dirs if not p.exists()]
if missing_dirs:
    print("Missing required L3 dataset directories:")
    for path in missing_dirs:
        print(f" - {path}")
else:
    print(f"L3 dataset layout looks good under: {data_root}")


def scan_image_directory(directory: Path) -> dict[str, Path]:
    if not directory.exists():
        raise FileNotFoundError(f"Missing directory: {directory}")
    return {
        path.stem: path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    }


def collect_split_pairs(hr_dir: Path, lr_dir: Path, split_name: str) -> list[tuple[Path, Path, str]]:
    hr_map = scan_image_directory(hr_dir)
    lr_map = scan_image_directory(lr_dir)
    pairs: list[tuple[Path, Path, str]] = []
    for stem in sorted(set(hr_map) & set(lr_map)):
        name = stem if split_name == "val" else f"{split_name}/{stem}"
        pairs.append((lr_map[stem], hr_map[stem], name))
    return pairs


def collect_all_pairs(data_root: Path) -> tuple[list[tuple[Path, Path, str]], list[tuple[Path, Path, str]]]:
    train_pairs: list[tuple[Path, Path, str]] = []
    for hr_rel, lr_rel, split_name in L3_TRAIN_SPLITS:
        train_pairs.extend(
            collect_split_pairs(
                data_root / hr_rel,
                data_root / lr_rel,
                split_name=split_name,
            )
        )
    val_pairs = collect_split_pairs(data_root / L3_VAL_DIRS[0], data_root / L3_VAL_DIRS[1], split_name="val")
    return train_pairs, val_pairs


class PairedSRDataset(Dataset):
    def __init__(self, pairs: list[tuple[Path, Path, str]], *, train: bool, patch_size: int, eval_size: int, seed: int):
        self.pairs = pairs
        self.train = train
        self.patch_size = patch_size
        self.eval_size = eval_size
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    @staticmethod
    def to_tensor(img: Image.Image) -> torch.Tensor:
        arr = np.asarray(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

    def augment_pair(self, lr: Image.Image, hr: Image.Image, index: int) -> tuple[Image.Image, Image.Image]:
        rng = random.Random((self.seed * 1_000_003) + (self.epoch * 9_973) + index)
        width, height = lr.size
        patch = self.patch_size
        if width < patch or height < patch:
            lr = lr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            hr = hr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            width, height = lr.size
        left = rng.randint(0, width - patch)
        top = rng.randint(0, height - patch)
        box = (left, top, left + patch, top + patch)
        lr = lr.crop(box)
        hr = hr.crop(box)
        if rng.random() < 0.5:
            lr = lr.transpose(FLIP_LEFT_RIGHT)
            hr = hr.transpose(FLIP_LEFT_RIGHT)
        if rng.random() < 0.5:
            lr = lr.transpose(FLIP_TOP_BOTTOM)
            hr = hr.transpose(FLIP_TOP_BOTTOM)
        rot = rng.choice([0, 1, 2, 3])
        if rot:
            rotate_ops = [ROTATE_90, ROTATE_180, ROTATE_270]
            lr = lr.transpose(rotate_ops[rot - 1])
            hr = hr.transpose(rotate_ops[rot - 1])
        return lr, hr

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> dict[str, Any]:
        lr_path, hr_path, name = self.pairs[index]
        lr = Image.open(lr_path).convert("RGB")
        hr = Image.open(hr_path).convert("RGB")
        if self.train:
            lr, hr = self.augment_pair(lr, hr, index)
        elif lr.size != (self.eval_size, self.eval_size):
            lr = lr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
            hr = hr.resize((self.eval_size, self.eval_size), BICUBIC_RESAMPLE)
        return {"lr": self.to_tensor(lr), "hr": self.to_tensor(hr), "name": name}


train_pairs, val_pairs = collect_all_pairs(data_root)
if cfg.train_pair_limit is not None:
    train_pairs = train_pairs[: cfg.train_pair_limit]
if cfg.val_pair_limit is not None:
    val_pairs = val_pairs[: cfg.val_pair_limit]
if not train_pairs or not val_pairs:
    raise RuntimeError(f"Expected non-empty train/val pairs, found train={len(train_pairs)} val={len(val_pairs)}")
if len(train_pairs) != EXPECTED_TRAIN_PAIRS and cfg.train_pair_limit is None:
    print(f"Warning: expected {EXPECTED_TRAIN_PAIRS} train pairs, found {len(train_pairs)}")
if len(val_pairs) != EXPECTED_VAL_PAIRS and cfg.val_pair_limit is None:
    print(f"Warning: expected {EXPECTED_VAL_PAIRS} val pairs, found {len(val_pairs)}")

train_eval_pairs = train_pairs[: min(cfg.train_eval_subset_size, len(train_pairs))]
train_eval_sample_names = [name for _, _, name in train_eval_pairs]

train_ds = PairedSRDataset(train_pairs, train=True, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size, seed=cfg.seed)
train_eval_ds = PairedSRDataset(train_eval_pairs, train=False, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size, seed=cfg.seed)
val_ds = PairedSRDataset(val_pairs, train=False, patch_size=cfg.train_patch_size, eval_size=cfg.eval_size, seed=cfg.seed)

pin_memory = device.type == "cuda"
loader_kwargs = {"num_workers": int(cfg.num_workers), "pin_memory": pin_memory}
if cfg.num_workers > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = cfg.prefetch_factor

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, **loader_kwargs)
train_eval_loader = DataLoader(train_eval_ds, batch_size=cfg.batch_size, shuffle=False, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, **loader_kwargs)

sample = next(iter(val_loader))
assert tuple(sample["lr"].shape[1:]) == (3, 256, 256)
assert tuple(sample["hr"].shape[1:]) == (3, 256, 256)
print({
    "train_pairs": len(train_pairs),
    "val_pairs": len(val_pairs),
    "train_eval_subset_size": len(train_eval_pairs),
    "train_eval_sample_names": train_eval_sample_names[:5],
    "train_batches_per_epoch": len(train_loader),
    "train_eval_batches": len(train_eval_loader),
    "val_batches_per_epoch": len(val_loader),
    "train_sample_shape": tuple(sample["lr"].shape),
    "device": str(device),
    "run_mode": cfg.run_mode,
})


L3 dataset layout looks good under: /content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/Data
{'train_pairs': 2217, 'val_pairs': 110, 'train_eval_subset_size': 128, 'train_eval_sample_names': ['L3_HR_train1/0000', 'L3_HR_train1/0001', 'L3_HR_train1/00010', 'L3_HR_train1/000100', 'L3_HR_train1/000101'], 'train_batches_per_epoch': 278, 'train_eval_batches': 16, 'val_batches_per_epoch': 14, 'train_sample_shape': (8, 3, 256, 256), 'device': 'cuda', 'run_mode': 'full'}


## 3. Model and Training

A norm-free mixed-kernel residual U-Net, residual-target Charbonnier+L1 loss, warmup+cosine LR, EMA validation, and full-run checkpointing.


In [4]:
@dataclass
class MixedKernelUNetResidualConfig:
    channels: int = 48
    encoder_blocks: tuple[int, int] = (3, 3)
    bottleneck_blocks: int = 6
    decoder_blocks: tuple[int, int] = (3, 3)
    kernel_pattern: tuple[int, ...] = (3, 5, 3, 5, 3, 5)
    residual_scale: float = 0.2
    global_residual: bool = True


def init_tail_small(conv: nn.Conv2d, scale: float = 1e-3) -> None:
    nn.init.kaiming_normal_(conv.weight, a=0.1, mode="fan_in", nonlinearity="leaky_relu")
    conv.weight.data.mul_(scale)
    if conv.bias is not None:
        nn.init.zeros_(conv.bias)


class MixedKernelResidualBlock(nn.Module):
    def __init__(self, channels: int, kernel_size: int, residual_scale: float = 0.2):
        super().__init__()
        padding = kernel_size // 2
        self.residual_scale = residual_scale
        self.conv1 = nn.Conv2d(channels, channels, kernel_size, 1, padding)
        self.act = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size, 1, padding)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.conv2(self.act(self.conv1(x)))
        return x + self.residual_scale * residual


class DownsampleBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=2, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UpsampleBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


# x -> stem -> enc1 -> down1 -> enc2 -> down2 -> bottleneck -> up1 + e2 -> dec1 -> up2 + e1 -> dec2 -> tail -> x + delta
class MixedKernelUNetResidualSR(nn.Module):
    def __init__(self, cfg_m: MixedKernelUNetResidualConfig):
        super().__init__()
        c = cfg_m.channels
        pattern = cfg_m.kernel_pattern
        self.global_residual = cfg_m.global_residual
        self.stem = nn.Sequential(nn.Conv2d(3, c, 3, 1, 1), nn.LeakyReLU(0.1, inplace=True))
        self.enc1 = nn.Sequential(*[
            MixedKernelResidualBlock(c, pattern[idx % len(pattern)], cfg_m.residual_scale)
            for idx in range(cfg_m.encoder_blocks[0])
        ])
        self.down1 = DownsampleBlock(c, c * 2)
        self.enc2 = nn.Sequential(*[
            MixedKernelResidualBlock(c * 2, pattern[(cfg_m.encoder_blocks[0] + idx) % len(pattern)], cfg_m.residual_scale)
            for idx in range(cfg_m.encoder_blocks[1])
        ])
        self.down2 = DownsampleBlock(c * 2, c * 4)
        self.bottleneck = nn.Sequential(*[
            MixedKernelResidualBlock(c * 4, pattern[(sum(cfg_m.encoder_blocks) + idx) % len(pattern)], cfg_m.residual_scale)
            for idx in range(cfg_m.bottleneck_blocks)
        ])
        self.up1 = UpsampleBlock(c * 4, c * 2)
        self.dec1 = nn.Sequential(*[
            MixedKernelResidualBlock(c * 2, pattern[(sum(cfg_m.encoder_blocks) + cfg_m.bottleneck_blocks + idx) % len(pattern)], cfg_m.residual_scale)
            for idx in range(cfg_m.decoder_blocks[0])
        ])
        self.up2 = UpsampleBlock(c * 2, c)
        self.dec2 = nn.Sequential(*[
            MixedKernelResidualBlock(c, pattern[(sum(cfg_m.encoder_blocks) + cfg_m.bottleneck_blocks + cfg_m.decoder_blocks[0] + idx) % len(pattern)], cfg_m.residual_scale)
            for idx in range(cfg_m.decoder_blocks[1])
        ])
        self.tail = nn.Conv2d(c, 3, 3, 1, 1)
        init_tail_small(self.tail)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = self.stem(x)
        e1 = self.enc1(s)
        d1 = self.down1(e1)
        e2 = self.enc2(d1)
        d2 = self.down2(e2)
        b = self.bottleneck(d2)
        u1 = self.up1(b)
        u1 = self.dec1(u1 + e2)
        u2 = self.up2(u1)
        u2 = self.dec2(u2 + e1)
        delta = self.tail(u2)
        return x + delta if self.global_residual else delta


class LegacyHalfInstanceResidualBlock(nn.Module):
    def __init__(self, channels: int, residual_scale: float = 0.2):
        super().__init__()
        self.residual_scale = residual_scale
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1)
        self.inorm = nn.InstanceNorm2d(channels // 2, affine=True)
        self.act = nn.LeakyReLU(0.1, inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.act(self.conv1(x))
        y_norm, y_pass = torch.chunk(y, 2, dim=1)
        y = torch.cat((self.inorm(y_norm), y_pass), dim=1)
        return x + self.residual_scale * self.conv2(y)


class LegacyResidualBlock(nn.Module):
    def __init__(self, channels: int, residual_scale: float = 0.2):
        super().__init__()
        self.residual_scale = residual_scale
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(channels, channels, 3, 1, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.residual_scale * self.block(x)


class LegacyHiNetLiteResidualSR(nn.Module):
    def __init__(self, cfg_m: MixedKernelUNetResidualConfig):
        super().__init__()
        c = cfg_m.channels
        self.global_residual = cfg_m.global_residual
        self.stem = nn.Sequential(nn.Conv2d(3, c, 3, 1, 1), nn.LeakyReLU(0.1, inplace=True))
        self.enc1 = nn.Sequential(*[LegacyHalfInstanceResidualBlock(c, cfg_m.residual_scale) for _ in range(cfg_m.encoder_blocks[0])])
        self.down1 = DownsampleBlock(c, c * 2)
        self.enc2 = nn.Sequential(*[LegacyHalfInstanceResidualBlock(c * 2, cfg_m.residual_scale) for _ in range(cfg_m.encoder_blocks[1])])
        self.down2 = DownsampleBlock(c * 2, c * 4)
        self.bottleneck = nn.Sequential(*[LegacyResidualBlock(c * 4, cfg_m.residual_scale) for _ in range(cfg_m.bottleneck_blocks)])
        self.up1 = UpsampleBlock(c * 4, c * 2)
        self.dec1 = nn.Sequential(*[LegacyResidualBlock(c * 2, cfg_m.residual_scale) for _ in range(cfg_m.decoder_blocks[0])])
        self.up2 = UpsampleBlock(c * 2, c)
        self.dec2 = nn.Sequential(*[LegacyResidualBlock(c, cfg_m.residual_scale) for _ in range(cfg_m.decoder_blocks[1])])
        self.tail = nn.Conv2d(c, 3, 3, 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s = self.stem(x)
        e1 = self.enc1(s)
        d1 = self.down1(e1)
        e2 = self.enc2(d1)
        d2 = self.down2(e2)
        b = self.bottleneck(d2)
        u1 = self.up1(b)
        u1 = self.dec1(u1 + e2)
        u2 = self.up2(u1)
        u2 = self.dec2(u2 + e1)
        delta = self.tail(u2)
        return x + delta if self.global_residual else delta


def detect_checkpoint_architecture_from_state(state_dict: dict[str, Any] | None) -> str:
    if state_dict and any(".inorm." in key for key in state_dict):
        return "legacy_half_instance_unet_residual"
    return "mixed_kernel_unet_residual_v3"


def detect_checkpoint_architecture_from_payload(checkpoint: dict[str, Any]) -> str:
    state_dict = checkpoint.get("model") or checkpoint.get("ema_model")
    return detect_checkpoint_architecture_from_state(state_dict)


def detect_model_architecture_for_run(resume_checkpoint_path: str | None) -> str:
    if not resume_checkpoint_path:
        return "mixed_kernel_unet_residual_v3"
    checkpoint = torch.load(resume_checkpoint_path, map_location="cpu")
    return detect_checkpoint_architecture_from_payload(checkpoint)


def build_sr_model(cfg_m: MixedKernelUNetResidualConfig, architecture: str) -> nn.Module:
    if architecture == "legacy_half_instance_unet_residual":
        return LegacyHiNetLiteResidualSR(cfg_m)
    if architecture == "mixed_kernel_unet_residual_v3":
        return MixedKernelUNetResidualSR(cfg_m)
    raise ValueError(f"Unknown model architecture: {architecture}")


def count_parameters(module: nn.Module) -> dict[str, int]:
    total = sum(param.numel() for param in module.parameters())
    trainable = sum(param.numel() for param in module.parameters() if param.requires_grad)
    return {"total": int(total), "trainable": int(trainable)}


def load_model_config_for_run(resume_checkpoint_path: str | None) -> MixedKernelUNetResidualConfig:
    base = asdict(MixedKernelUNetResidualConfig())
    if resume_checkpoint_path:
        checkpoint = torch.load(resume_checkpoint_path, map_location="cpu")
        checkpoint_model_config = checkpoint.get("model_config") or {}
        for key in base:
            if key in checkpoint_model_config:
                value = checkpoint_model_config[key]
                if key in {"encoder_blocks", "decoder_blocks", "kernel_pattern"}:
                    value = tuple(value)
                base[key] = value
        print({
            "loaded_model_config_from_checkpoint": resume_checkpoint_path,
            "checkpoint_model_config": checkpoint_model_config,
        })
    return MixedKernelUNetResidualConfig(**base)


model_cfg = load_model_config_for_run(cfg.resume_checkpoint_path if cfg.enable_resume else None)
model_architecture = detect_model_architecture_for_run(cfg.resume_checkpoint_path if cfg.enable_resume else None)
model = build_sr_model(model_cfg, model_architecture).to(device)
if cfg.use_channels_last and device.type == "cuda":
    model = model.to(memory_format=torch.channels_last)
shape_input = torch.randn(1, 3, 256, 256, device=device)
if cfg.use_channels_last and device.type == "cuda":
    shape_input = shape_input.contiguous(memory_format=torch.channels_last)
shape_check = model(shape_input).shape
assert tuple(shape_check) == (1, 3, 256, 256), f"Unexpected shape: {shape_check}"
model_num_parameters = count_parameters(model)
print({
    "model_variant": model_architecture,
    "shape_check": tuple(shape_check),
    "model_num_parameters": model_num_parameters,
})


{'loaded_model_config_from_checkpoint': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_colab_v3_full_20260423_234240/checkpoints/last.pth', 'checkpoint_model_config': {'channels': 48, 'encoder_blocks': (3, 3), 'bottleneck_blocks': 6, 'decoder_blocks': (3, 3), 'kernel_pattern': (3, 5, 3, 5, 3, 5), 'residual_scale': 0.2, 'global_residual': True}}
{'model_variant': 'mixed_kernel_unet_residual_v3', 'shape_check': (1, 3, 256, 256), 'model_num_parameters': {'total': 10453443, 'trainable': 10453443}}


In [5]:
def psnr_from_mse(mse: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return 10.0 * torch.log10(1.0 / torch.clamp(mse, min=eps))


@torch.no_grad()
def batch_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    mse = F.mse_loss(pred, target, reduction="none").mean(dim=(1, 2, 3))
    return float(psnr_from_mse(mse).mean().item())


@torch.inference_mode()
def compute_input_psnr(loader: DataLoader, device: torch.device) -> float:
    meter = 0.0
    count = 0
    for batch in loader:
        lr = move_image_batch(batch["lr"], device)
        hr = move_image_batch(batch["hr"], device)
        meter += batch_psnr(lr, hr)
        count += 1
    return meter / max(count, 1)


def charbonnier_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    diff = pred - target
    return torch.sqrt(diff * diff + eps * eps).mean()


def compute_restoration_loss(pred: torch.Tensor, lr: torch.Tensor, hr: torch.Tensor) -> dict[str, torch.Tensor]:
    pred_res = pred - lr
    target_res = hr - lr
    loss_char = charbonnier_loss(pred_res, target_res)
    loss_l1 = F.l1_loss(pred_res, target_res)
    total = 0.7 * loss_char + 0.3 * loss_l1
    return {
        "loss": total,
        "loss_charbonnier": loss_char.detach(),
        "loss_l1": loss_l1.detach(),
    }


def lr_at_epoch(epoch_idx: int, *, base_lr: float, min_lr: float, warmup_epochs: int, total_epochs: int) -> float:
    step = epoch_idx + 1
    if step <= warmup_epochs:
        return base_lr * step / max(1, warmup_epochs)
    progress = (step - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_lr + cosine * (base_lr - min_lr)


class EMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for key, value in model.state_dict().items():
            self.shadow[key].mul_(self.decay).add_(value.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def copy_to(self, model: nn.Module) -> None:
        model.load_state_dict(self.shadow, strict=True)


def move_image_batch(batch: torch.Tensor, device: torch.device) -> torch.Tensor:
    batch = batch.to(device, non_blocking=(device.type == "cuda"))
    if cfg.use_channels_last and device.type == "cuda":
        batch = batch.contiguous(memory_format=torch.channels_last)
    return batch


def current_lr(optimizer: torch.optim.Optimizer) -> float:
    return float(optimizer.param_groups[0]["lr"])


def infer_resume_run_root(checkpoint_path: str | None) -> Path | None:
    if not checkpoint_path:
        return None
    ckpt_path = Path(checkpoint_path)
    if ckpt_path.parent.name != "checkpoints":
        return None
    return ckpt_path.parent.parent


def get_scaler_state(scaler) -> dict[str, Any] | None:
    return scaler.state_dict() if scaler.is_enabled() else None


def infer_best_checkpoint_path(checkpoint_path: str | None) -> Path | None:
    run_root = infer_resume_run_root(checkpoint_path)
    if run_root is None:
        return Path(checkpoint_path) if checkpoint_path else None
    checkpoints = run_root / "checkpoints"
    for name in ("best.pth", "best_ema.pth"):
        candidate = checkpoints / name
        if candidate.exists():
            return candidate
    return Path(checkpoint_path) if checkpoint_path else None


def carry_forward_best_checkpoint(resume_checkpoint_path: str | None, destination: Path) -> str | None:
    source = infer_best_checkpoint_path(resume_checkpoint_path)
    if source is None or not source.exists():
        return None
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.resolve() != destination.resolve():
        shutil.copy2(source, destination)
    return str(source)


def normalize_history_row(row: dict[str, Any]) -> dict[str, Any]:
    train_eval_psnr = row.get("train_eval_psnr")
    val_psnr = row.get("val_psnr")
    gap = row.get("generalization_gap")
    if gap is None and train_eval_psnr not in (None, "") and val_psnr not in (None, ""):
        gap = float(train_eval_psnr) - float(val_psnr)
    return {
        "epoch": int(row["epoch"]),
        "epoch_time_sec": float(row["epoch_time_sec"]),
        "lr": float(row["lr"]),
        "train_loss_residual": float(row["train_loss_residual"]),
        "train_patch_psnr": float(row["train_patch_psnr"]),
        "train_eval_psnr": float(train_eval_psnr),
        "val_psnr": float(val_psnr),
        "input_psnr": float(row["input_psnr"]),
        "delta_psnr": float(row["delta_psnr"]),
        "generalization_gap": float(gap),
        "is_best": bool(row.get("is_best", False)),
    }


def load_resume_history_rows(checkpoint_path: str | None) -> list[dict[str, Any]]:
    run_root = infer_resume_run_root(checkpoint_path)
    if run_root is None:
        return []
    history_path = run_root / "history.json"
    if history_path.exists():
        payload = json.loads(history_path.read_text(encoding="utf-8"))
        return [normalize_history_row(row) for row in payload.get("epochs", [])]
    metrics_path = run_root / "metrics.csv"
    if not metrics_path.exists():
        return []
    with metrics_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return [normalize_history_row(row) for row in reader]


def write_history_json(history_path: Path, rows: list[dict[str, Any]], resume_checkpoint_path: str | None) -> None:
    payload = {
        "resume_checkpoint_path": resume_checkpoint_path,
        "epoch_count": len(rows),
        "latest_epoch": int(rows[-1]["epoch"]) if rows else 0,
        "best_val_psnr": max((float(row["val_psnr"]) for row in rows), default=None),
        "epochs": rows,
    }
    history_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def print_epoch_row(row: dict[str, Any]) -> None:
    print(
        f"[epoch {int(row['epoch']):03d}] "
        f"time={float(row['epoch_time_sec']):.2f}s "
        f"lr={float(row['lr']):.6f} "
        f"train_patch_psnr={float(row['train_patch_psnr']):.4f} "
        f"train_eval_psnr={float(row['train_eval_psnr']):.4f} "
        f"val_psnr={float(row['val_psnr']):.4f} "
        f"generalization_gap={float(row['generalization_gap']):.4f} "
        f"is_best={bool(row.get('is_best', False))}"
    )


def build_export_model_from_checkpoint(
    checkpoint: dict[str, Any],
    model_cfg: MixedKernelUNetResidualConfig,
    device: torch.device,
    weight_source: str,
) -> nn.Module:
    export_architecture = detect_checkpoint_architecture_from_payload(checkpoint)
    export_model = build_sr_model(model_cfg, export_architecture).to(device).eval()
    if weight_source == "ema":
        ema_state = checkpoint.get("ema_model")
        if ema_state is None:
            raise ValueError("Checkpoint is missing ema_model for selected_weight_source='ema'")
        export_model.load_state_dict(ema_state, strict=True)
    else:
        export_model.load_state_dict(checkpoint["model"], strict=True)
    if cfg.use_channels_last and device.type == "cuda":
        export_model = export_model.to(memory_format=torch.channels_last)
    return export_model


def load_resume_checkpoint(
    checkpoint_path: str | None,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler,
    ema: EMA | None,
    device: torch.device,
    strict: bool,
) -> tuple[int, float, int, dict[str, Any] | None]:
    if not checkpoint_path:
        return 0, -math.inf, 0, None
    ckpt_path = Path(checkpoint_path)
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model"], strict=strict)
    if checkpoint.get("optimizer") is not None:
        optimizer.load_state_dict(checkpoint["optimizer"])
    if ema is not None and checkpoint.get("ema_model") is not None:
        ema.shadow = {k: v.detach().clone().to(device) for k, v in checkpoint["ema_model"].items()}
    scaler_state = checkpoint.get("scaler")
    if scaler_state and scaler.is_enabled():
        scaler.load_state_dict(scaler_state)
    return (
        int(checkpoint.get("epoch", 0)),
        float(checkpoint.get("best_val_psnr", -math.inf)),
        int(checkpoint.get("best_epoch", 0)),
        checkpoint.get("last_metrics"),
    )


def make_checkpoint_payload(
    *,
    epoch: int,
    model: nn.Module,
    ema: EMA | None,
    optimizer: torch.optim.Optimizer,
    scaler,
    last_metrics: dict[str, Any],
    best_val_psnr: float,
    best_epoch: int,
) -> dict[str, Any]:
    return {
        "epoch": int(epoch),
        "model": model.state_dict(),
        "ema_model": (ema.shadow if ema is not None else None),
        "optimizer": optimizer.state_dict(),
        "scaler": get_scaler_state(scaler),
        "config": asdict(cfg),
        "model_config": asdict(model_cfg),
        "selected_weight_source": cfg.selected_weight_source,
        "best_metric_name": "val_psnr",
        "best_val_psnr": float(best_val_psnr),
        "best_epoch": int(best_epoch),
        "last_metrics": last_metrics,
    }


@torch.inference_mode()
def evaluate_psnr(model: nn.Module, loader: DataLoader, device: torch.device) -> dict[str, float]:
    model.eval()
    pred_meter = 0.0
    input_meter = 0.0
    count = 0
    for batch in loader:
        lr = move_image_batch(batch["lr"], device)
        hr = move_image_batch(batch["hr"], device)
        pred = model(lr).clamp(0, 1)
        pred_meter += batch_psnr(pred, hr)
        input_meter += batch_psnr(lr, hr)
        count += 1
    val_psnr = pred_meter / max(count, 1)
    input_psnr = input_meter / max(count, 1)
    return {
        "val_psnr": float(val_psnr),
        "input_psnr": float(input_psnr),
        "delta_psnr": float(val_psnr - input_psnr),
    }


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler,
    ema: EMA | None,
    device: torch.device,
) -> tuple[float, float]:
    model.train()
    amp_enabled = bool(cfg.use_amp and device.type == "cuda")
    loss_meter = 0.0
    psnr_meter = 0.0
    count = 0
    for batch in loader:
        lr = move_image_batch(batch["lr"], device)
        hr = move_image_batch(batch["hr"], device)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context(device.type, amp_enabled):
            pred = model(lr)
            loss_info = compute_restoration_loss(pred, lr, hr)
            loss = loss_info["loss"]
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()
        if ema is not None:
            ema.update(model)
        with torch.no_grad():
            loss_meter += float(loss.item())
            psnr_meter += batch_psnr(pred.clamp(0, 1), hr)
            count += 1
    return loss_meter / max(count, 1), psnr_meter / max(count, 1)


validation_input_psnr = compute_input_psnr(val_loader, device)
print({"validation_input_psnr": validation_input_psnr})

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = make_grad_scaler(enabled=(cfg.use_amp and device.type == "cuda"))
ema = EMA(model, decay=cfg.ema_decay) if cfg.use_ema else None
ema_eval_model = None
if ema is not None:
    ema_eval_model = build_sr_model(model_cfg, model_architecture).to(device)
    if cfg.use_channels_last and device.type == "cuda":
        ema_eval_model = ema_eval_model.to(memory_format=torch.channels_last)

metrics_header = [
    "epoch",
    "epoch_time_sec",
    "lr",
    "train_loss_residual",
    "train_patch_psnr",
    "train_eval_psnr",
    "val_psnr",
    "input_psnr",
    "delta_psnr",
    "generalization_gap",
    "is_best",
]

best_ckpt_path = checkpoints_dir / "best.pth"
last_ckpt_path = checkpoints_dir / "last.pth"
resume_checkpoint_path = cfg.resume_checkpoint_path if cfg.enable_resume else None
carried_best_checkpoint_path = carry_forward_best_checkpoint(resume_checkpoint_path, best_ckpt_path)
if carried_best_checkpoint_path:
    print({"carried_forward_best_checkpoint": carried_best_checkpoint_path, "new_best_checkpoint_path": str(best_ckpt_path)})
start_epoch, best_val_psnr, best_epoch, last_metrics = load_resume_checkpoint(
    resume_checkpoint_path,
    model,
    optimizer,
    scaler,
    ema,
    device,
    strict=cfg.resume_strict,
)
resume_history_rows = load_resume_history_rows(resume_checkpoint_path)
history_rows = [dict(row) for row in resume_history_rows]
resume_baseline_best_val_psnr = float(best_val_psnr)
resume_baseline_best_epoch = int(best_epoch)
stop_reason = "completed_max_epochs"

with metrics_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=metrics_header)
    writer.writeheader()
    for row in resume_history_rows:
        writer.writerow(row)
with epoch_logs_jsonl_path.open("w", encoding="utf-8") as f:
    for row in resume_history_rows:
        f.write(json.dumps(row) + "\n")
write_history_json(history_json_path, history_rows, resume_checkpoint_path)

if resume_history_rows:
    print("Replaying prior run history from history.json/metrics.csv")
    for row in resume_history_rows:
        print_epoch_row(row)
elif last_metrics is not None:
    print("No metrics.csv found for resume run; using checkpoint last_metrics")
    print_epoch_row(normalize_history_row(last_metrics))

for epoch in range(start_epoch, cfg.epochs):
    train_ds.set_epoch(epoch)
    epoch_lr = lr_at_epoch(
        epoch,
        base_lr=cfg.lr,
        min_lr=cfg.min_lr,
        warmup_epochs=cfg.warmup_epochs,
        total_epochs=cfg.epochs,
    )
    for group in optimizer.param_groups:
        group["lr"] = epoch_lr

    t0 = time.perf_counter()
    train_loss_residual, train_patch_psnr = train_one_epoch(model, train_loader, optimizer, scaler, ema, device)

    eval_model = model
    if cfg.selected_weight_source == "ema":
        if ema is None or ema_eval_model is None:
            raise RuntimeError("EMA evaluation requested but EMA is disabled")
        ema.copy_to(ema_eval_model)
        eval_model = ema_eval_model

    val_metrics = evaluate_psnr(eval_model, val_loader, device)
    train_eval_metrics = evaluate_psnr(eval_model, train_eval_loader, device)
    epoch_time_sec = float(time.perf_counter() - t0)
    generalization_gap = float(train_eval_metrics["val_psnr"] - val_metrics["val_psnr"])
    is_best = bool(val_metrics["val_psnr"] > best_val_psnr)
    if is_best:
        best_val_psnr = float(val_metrics["val_psnr"])
        best_epoch = int(epoch + 1)

    row = {
        "epoch": int(epoch + 1),
        "epoch_time_sec": epoch_time_sec,
        "lr": current_lr(optimizer),
        "train_loss_residual": float(train_loss_residual),
        "train_patch_psnr": float(train_patch_psnr),
        "train_eval_psnr": float(train_eval_metrics["val_psnr"]),
        "val_psnr": float(val_metrics["val_psnr"]),
        "input_psnr": float(val_metrics["input_psnr"]),
        "delta_psnr": float(val_metrics["delta_psnr"]),
        "generalization_gap": generalization_gap,
        "is_best": is_best,
    }

    payload = make_checkpoint_payload(
        epoch=row["epoch"],
        model=model,
        ema=ema,
        optimizer=optimizer,
        scaler=scaler,
        last_metrics=row,
        best_val_psnr=best_val_psnr,
        best_epoch=best_epoch,
    )
    torch.save(payload, last_ckpt_path)
    if is_best:
        torch.save(payload, best_ckpt_path)

    with metrics_csv_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=metrics_header)
        writer.writerow(row)
    with epoch_logs_jsonl_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")
    history_rows.append(row)
    write_history_json(history_json_path, history_rows, resume_checkpoint_path)
    print_epoch_row(row)

    resumed_epochs_run = int(row["epoch"] - start_epoch)
    no_resume_best_after_limit = (
        resume_checkpoint_path is not None
        and cfg.stop_if_no_resume_best_after_epochs is not None
        and resumed_epochs_run >= int(cfg.stop_if_no_resume_best_after_epochs)
        and float(best_val_psnr) <= resume_baseline_best_val_psnr
    )
    if no_resume_best_after_limit:
        stop_reason = (
            f"stopped_no_improvement_over_resumed_best_after_{int(cfg.stop_if_no_resume_best_after_epochs)}_epochs"
        )
        print({
            "stop_reason": stop_reason,
            "resume_baseline_best_val_psnr": resume_baseline_best_val_psnr,
            "resume_baseline_best_epoch": resume_baseline_best_epoch,
            "current_best_val_psnr": best_val_psnr,
            "epochs_since_resume": resumed_epochs_run,
        })
        break

latest_epoch = int(history_rows[-1]["epoch"]) if history_rows else start_epoch
print({
    "best_epoch": best_epoch,
    "best_val_psnr": best_val_psnr,
    "best_ckpt_path": str(best_ckpt_path),
    "last_ckpt_path": str(last_ckpt_path),
    "carried_best_checkpoint_path": carried_best_checkpoint_path,
    "resume_baseline_best_val_psnr": resume_baseline_best_val_psnr,
    "resume_baseline_best_epoch": resume_baseline_best_epoch,
    "latest_epoch": latest_epoch,
    "stop_reason": stop_reason,
})


{'validation_input_psnr': 23.12304142543248}
{'carried_forward_best_checkpoint': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260423/hinet_lite_npu_v1_colab_v3_full_20260423_234240/checkpoints/best.pth', 'new_best_checkpoint_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260424/hinet_lite_npu_v1_colab_v3_full_20260424_004323/checkpoints/best.pth'}
Replaying prior run history from history.json/metrics.csv
[epoch 001] time=854.86s lr=0.000060 train_patch_psnr=24.3505 train_eval_psnr=24.4894 val_psnr=23.1279 generalization_gap=1.3615 is_best=True
[epoch 002] time=37.98s lr=0.000120 train_patch_psnr=24.5605 train_eval_psnr=24.5060 val_psnr=23.1419 generalization_gap=1.3641 is_best=True
[epoch 003] time=45.15s lr=0.000180 train_patch_psnr=24.7201 train_eval_psnr=24.5363 val_psnr=23.1678 generalization_gap=1.3685 is_best=True
[epoch 004] time=46.11s lr=0.000240 train_patch_psnr=24.8413 train_eval_psnr=24.5827 val_psnr=23.2094 gener

## 4. Validation

Best-checkpoint preview from the selected export state.


In [6]:
def tensor_to_uint8_image(x: torch.Tensor) -> Image.Image:
    x = x.detach().cpu().clamp(0, 1)
    arr = (x.permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
    return Image.fromarray(arr)


@torch.no_grad()
def write_val_preview(model: nn.Module, loader: DataLoader, out_dir: Path, device: torch.device) -> dict[str, str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    batch = next(iter(loader))
    lr = move_image_batch(batch["lr"], device)
    hr = move_image_batch(batch["hr"], device)
    name = batch["name"][0]
    pred = model(lr).clamp(0, 1)
    lr_path = out_dir / "val_preview_input.png"
    pred_path = out_dir / "val_preview_pred.png"
    hr_path = out_dir / "val_preview_target.png"
    tensor_to_uint8_image(lr[0]).save(lr_path)
    tensor_to_uint8_image(pred[0]).save(pred_path)
    tensor_to_uint8_image(hr[0]).save(hr_path)
    return {
        "sample_name": str(name),
        "input": str(lr_path),
        "prediction": str(pred_path),
        "target": str(hr_path),
    }


checkpoint = torch.load(best_ckpt_path, map_location=device)
checkpoint_weight_source = checkpoint.get("selected_weight_source", cfg.selected_weight_source)
export_model = build_export_model_from_checkpoint(checkpoint, model_cfg, device, checkpoint_weight_source)
preview_info = write_val_preview(export_model, val_loader, exports_dir / "val_preview", device)
print(preview_info)


{'sample_name': '0000', 'input': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260424/hinet_lite_npu_v1_colab_v3_full_20260424_004323/exports/val_preview/val_preview_input.png', 'prediction': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260424/hinet_lite_npu_v1_colab_v3_full_20260424_004323/exports/val_preview/val_preview_pred.png', 'target': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260424/hinet_lite_npu_v1_colab_v3_full_20260424_004323/exports/val_preview/val_preview_target.png'}


## 5. Export and Submission

ONNX export, parity check, training-derived calibration bundle, and final artifact summary.


In [7]:
onnx_path = exports_dir / f"{cfg.model_id}.onnx"
mxq_target_path = exports_dir / f"{cfg.model_id}.mxq"

dummy = torch.randn(1, 3, 256, 256, device=device)
if cfg.use_channels_last and device.type == "cuda":
    dummy = dummy.contiguous(memory_format=torch.channels_last)

torch.onnx.export(
    export_model,
    dummy,
    str(onnx_path),
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    dynamo=False,
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
node_ops = [node.op_type for node in onnx_model.graph.node]
node_ops_set = set(node_ops)
blocked_ops = [op for op in ["Softmax", "MatMul", "Mul", "Resize", "InstanceNormalization"] if op in node_ops_set]
op_audit = {
    "graph_op_count": len(node_ops),
    "blocked_ops_found": blocked_ops,
    "selected_weight_source": checkpoint_weight_source,
    "norm_free_graph": "InstanceNormalization" not in node_ops_set,
    "model_num_parameters": model_num_parameters,
}
print({"onnx_path": str(onnx_path), **op_audit})

parity_result = {
    "attempted": False,
    "onnxruntime_available": False,
    "passed": False,
    "max_abs_diff": None,
    "selected_weight_source": checkpoint_weight_source,
}
try:
    import onnxruntime as ort

    parity_result["attempted"] = True
    parity_result["onnxruntime_available"] = True
    cpu_model = build_export_model_from_checkpoint(checkpoint, model_cfg, torch.device("cpu"), checkpoint_weight_source).cpu().eval()
    parity_input = torch.randn(1, 3, 256, 256)
    with torch.no_grad():
        torch_out = cpu_model(parity_input).detach().cpu().numpy()
    ort_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    ort_out = ort_session.run(["output"], {"input": parity_input.numpy()})[0]
    max_abs_diff = float(np.max(np.abs(torch_out - ort_out)))
    parity_result["max_abs_diff"] = max_abs_diff
    parity_result["passed"] = bool(max_abs_diff < 1e-3)
    if not parity_result["passed"]:
        raise RuntimeError(f"ONNX parity failed: max_abs_diff={max_abs_diff:.6f}")
except ImportError:
    print("onnxruntime not installed; skipping CPU parity check")
print(parity_result)


def write_calibration_bundle(train_pairs: list[tuple[Path, Path, str]], out_dir: Path, max_items: int = 32) -> dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    selected = train_pairs[: max_items]
    manifest = {"source": "train_pairs_only", "count": len(selected), "items": []}
    for idx, (lr_path, _, name) in enumerate(selected):
        dst = out_dir / f"cal_{idx:04d}.png"
        Image.open(lr_path).convert("RGB").save(dst)
        manifest["items"].append({"name": name, "lr_source": str(lr_path), "cal_path": str(dst)})
    manifest_path = out_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return {"calibration_dir": str(out_dir), "manifest_path": str(manifest_path), "count": len(selected)}


cal_info = write_calibration_bundle(train_pairs, calibration_dir, max_items=32)
summary = {
    "model_id": cfg.model_id,
    "experiment_tag": cfg.experiment_tag,
    "model_variant": model_architecture,
    "model_num_parameters": model_num_parameters,
    "config": asdict(cfg),
    "model_config": asdict(model_cfg),
    "startup_mode": startup_mode,
    "resume": {
        "enabled": bool(cfg.enable_resume),
        "resume_checkpoint_path": resume_checkpoint_path,
        "carried_best_checkpoint_path": carried_best_checkpoint_path,
        "resume_history_replayed": bool(resume_history_rows),
        "start_epoch": int(start_epoch),
        "resume_baseline_best_val_psnr": resume_baseline_best_val_psnr,
        "resume_baseline_best_epoch": resume_baseline_best_epoch,
        "stop_if_no_resume_best_after_epochs": cfg.stop_if_no_resume_best_after_epochs,
    },
    "pair_summary": {
        "train_pairs": len(train_pairs),
        "val_pairs": len(val_pairs),
        "train_preview": [name for _, _, name in train_pairs[:3]],
        "val_preview": [name for _, _, name in val_pairs[:3]],
        "train_eval_subset_size": len(train_eval_pairs),
        "train_eval_sample_names": train_eval_sample_names,
    },
    "validation_input_psnr": float(validation_input_psnr),
    "loss_config": {
        "type": "residual_charbonnier_l1",
        "charbonnier_weight": 0.7,
        "l1_weight": 0.3,
    },
    "lr_schedule": {
        "type": "warmup_cosine",
        "warmup_epochs": cfg.warmup_epochs,
        "base_lr": cfg.lr,
        "min_lr": cfg.min_lr,
    },
    "grad_clip_norm": cfg.grad_clip_norm,
    "selected_weight_source": checkpoint_weight_source,
    "best_metric_name": "val_psnr",
    "stop_reason": stop_reason,
    "latest_epoch": int(latest_epoch),
    "best_val_psnr": float(best_val_psnr),
    "best_epoch": int(best_epoch),
    "metrics_csv_path": str(metrics_csv_path),
    "epoch_logs_jsonl_path": str(epoch_logs_jsonl_path),
    "history_json_path": str(history_json_path),
    "best_checkpoint_path": str(best_ckpt_path),
    "last_checkpoint_path": str(last_ckpt_path),
    "onnx_path": str(onnx_path),
    "op_audit": op_audit,
    "parity_result": parity_result,
    "calibration": cal_info,
    "mxq_target_path": str(mxq_target_path),
    "val_preview": preview_info,
    "latest_metrics": history_rows[-1] if history_rows else None,
}
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print({
    "summary_path": str(summary_path),
    "best_checkpoint": str(best_ckpt_path),
    "onnx": str(onnx_path),
    "model_num_parameters": model_num_parameters,
    "mxq_target": str(mxq_target_path),
    "stop_reason": stop_reason,
})


/tmp/ipykernel_4563/519675750.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


{'onnx_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260424/hinet_lite_npu_v1_colab_v3_full_20260424_004323/exports/hinet_lite_npu_v1.onnx', 'graph_op_count': 122, 'blocked_ops_found': ['Mul'], 'selected_weight_source': 'ema', 'norm_free_graph': True, 'model_num_parameters': {'total': 10453443, 'trainable': 10453443}}
onnxruntime not installed; skipping CPU parity check
{'attempted': False, 'onnxruntime_available': False, 'passed': False, 'max_abs_diff': None, 'selected_weight_source': 'ema'}
{'summary_path': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260424/hinet_lite_npu_v1_colab_v3_full_20260424_004323/summary.json', 'best_checkpoint': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260424/hinet_lite_npu_v1_colab_v3_full_20260424_004323/checkpoints/best.pth', 'onnx': '/content/drive/MyDrive/Data 255 Class Spring 2026/Data 255/Lab 3/runs/20260424/hinet_lite_npu_v1_colab_v3_full_20260424_004323/ex